<a href="https://colab.research.google.com/github/syedmahmoodiagents/transformers/blob/main/Full_maskattention_nextword.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import math
import spacy
import torch
import torch.nn as nn

In [2]:
torch.manual_seed(42)

In [3]:
nlp = spacy.load("en_core_web_sm")

In [4]:
nlp("I love doing mathematics")

I love doing mathematics

In [5]:
corpus = [
    "I love doing mathematics",
    "calculus is great mathematics",
    "mathematics is quite useful"
]

### for practice only

In [13]:
ndt = {"abc":"python", "2":"java"}

In [15]:
ndt["abc"] = "javascript"

In [16]:
ndt

{'abc': 'javascript', '2': 'java'}

In [17]:
ndt["three"] = "computer"

In [18]:
ndt

{'abc': 'javascript', '2': 'java', 'three': 'computer'}

In [25]:
dt={}
k = 0
for s in corpus:
    for tok in nlp(s):
        w=tok.text.lower()
        # dt[k] = w
        dt[w] = k
        k = k+1
        # ls.append(w)

In [26]:
dt

{'i': 0,
 'love': 1,
 'doing': 2,
 'mathematics': 8,
 'calculus': 4,
 'is': 9,
 'great': 6,
 'quite': 10,
 'useful': 11}

# practice finished

In [27]:
word2idx={"<PAD>":0,"<UNK>":1}
# idx2word={0:"<PAD>",1:"<UNK>"}
idx=2
for s in corpus:
    for tok in nlp(s):
        w=tok.text.lower()
        if w not in word2idx:
            word2idx[w]=idx
            # idx2word[idx]=w
            idx+=1

vocab_size=len(word2idx)

In [28]:
word2idx

{'<PAD>': 0,
 '<UNK>': 1,
 'i': 2,
 'love': 3,
 'doing': 4,
 'mathematics': 5,
 'calculus': 6,
 'is': 7,
 'great': 8,
 'quite': 9,
 'useful': 10}

In [23]:
X=[]
Y=[]
for s in corpus:
    ids=[word2idx[t.text.lower()] for t in nlp(s)]
    for i in range(1,len(ids)):
        X.append(ids[:i])
        Y.append(ids[i])

In [24]:
X

[[2], [2, 3], [2, 3, 4], [6], [6, 7], [6, 7, 8], [5], [5, 7], [5, 7, 9]]

In [29]:
Y

[3, 4, 5, 7, 8, 5, 7, 9, 10]

In [ ]:
max_len=max(len(x) for x in X)

In [ ]:
max_len

3

In [ ]:
for i in range(len(X)):
    X[i]=[0]*(max_len-len(X[i]))+X[i]

In [ ]:
X

[[0, 0, 2],
 [0, 2, 3],
 [2, 3, 4],
 [0, 0, 6],
 [0, 6, 7],
 [6, 7, 8],
 [0, 0, 5],
 [0, 5, 7],
 [5, 7, 9]]

In [ ]:
X=torch.tensor(X,dtype=torch.long)
Y=torch.tensor(Y,dtype=torch.long)

In [ ]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model

    def forward(self, x):

        batch_size, seq_len, _ = x.shape
        pe = torch.zeros(seq_len, self.d_model)
        position = torch.arange(seq_len).unsqueeze(1)

        div = torch.exp(torch.arange(0, self.d_model, 2) * (-math.log(10000.0) / self.d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)

        pe = pe.unsqueeze(0)

        return x + pe

In [ ]:
class MaskedSelfAttentionLM(nn.Module):

    def __init__(self,vocab_size,d_model):

        super().__init__()
        self.embedding=nn.Embedding(vocab_size,d_model)
        self.position=PositionalEncoding(d_model)

        self.query=nn.Linear(d_model,d_model)
        self.key=nn.Linear(d_model,d_model)
        self.value=nn.Linear(d_model,d_model)

        self.fc=nn.Linear(d_model,vocab_size)

    def forward(self,x):

        x=self.embedding(x)
        x=self.position(x)

        Q=self.query(x)
        K=self.key(x)
        V=self.value(x)

        # print(Q.shape, K.transpose(-2, -1))

        scores=torch.matmul(Q, K.transpose(-2,-1))
        scores=scores/math.sqrt(Q.size(-1))

        T=x.size(1)

        mask=torch.triu(torch.ones(T, T), diagonal=1).bool()
        scores=scores.masked_fill(mask, float("-inf"))

        weights=torch.softmax(scores,dim=-1)

        context=torch.matmul(weights, V)
        last_context=context[:,-1,:]

        logits=self.fc(last_context)
        return logits,weights

In [ ]:
model=MaskedSelfAttentionLM(vocab_size=vocab_size, d_model=32)

criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.01)

In [ ]:
epochs=300

for epoch in range(epochs):
    optimizer.zero_grad()
    logits,weights=model(X)
    loss=criterion(logits,Y)
    loss.backward()
    optimizer.step()

    if (epoch+1)%20==0:
        print(f"Epoch {epoch+1} Loss={loss.item():.4f}")

Epoch 20 Loss=0.3793
Epoch 40 Loss=0.1048
Epoch 60 Loss=0.0215
Epoch 80 Loss=0.0080
Epoch 100 Loss=0.0047
Epoch 120 Loss=0.0033
Epoch 140 Loss=0.0024
Epoch 160 Loss=0.0019
Epoch 180 Loss=0.0015
Epoch 200 Loss=0.0013
Epoch 220 Loss=0.0011
Epoch 240 Loss=0.0009
Epoch 260 Loss=0.0008
Epoch 280 Loss=0.0007
Epoch 300 Loss=0.0006


## Inference

In [ ]:
word2idx

{'<PAD>': 0,
 '<UNK>': 1,
 'i': 2,
 'love': 3,
 'doing': 4,
 'mathematics': 5,
 'calculus': 6,
 'is': 7,
 'great': 8,
 'quite': 9,
 'useful': 10}

In [ ]:
word2idx.get('bell', 1)

1

In [ ]:
idx2word

{0: '<PAD>',
 1: '<UNK>',
 2: 'i',
 3: 'love',
 4: 'doing',
 5: 'mathematics',
 6: 'calculus',
 7: 'is',
 8: 'great',
 9: 'quite',
 10: 'useful'}

In [ ]:

def predict(text):

    ids=[word2idx.get(t.text.lower(), 1) for t in nlp(text)]

    if len(ids)<max_len:
        ids=[0]*(max_len-len(ids))+ids
    else:
        ids=ids[-max_len:]

    inp = torch.tensor([ids])

    # model.eval() # Batch Operations
    # with torch.no_grad(): prevent from weights gtting change

    logits, weights = model(inp)
    pred = torch.argmax(logits, dim=1).item()

    # print("\nAttention Matrix")
    # print(weights.squeeze(0))

    return idx2word[pred]

In [ ]:


tests=[
    "I",
    "I love",
    "I love doing",
    "calculus",
    "calculus is",
    "calculus is great",
    "mathematics",
    "mathematics is",
    "mathematics is quite"
]

print("\nPredictions\n")

# for t in tests:
#     predict(t)

for t in tests:
    print(f"{t:25s} -> {predict(t)}")


Predictions

I                         -> love
I love                    -> doing
I love doing              -> mathematics
calculus                  -> is
calculus is               -> great
calculus is great         -> mathematics
mathematics               -> is
mathematics is            -> quite
mathematics is quite      -> useful


In [ ]:


tests=[
    "I",
    "I love",
    "I love doing",
    "calculus",
    "calculus is",
    "calculus is great",
    "mathematics",
    "mathematics is",
    "mathematics is quite"
]

print("\nPredictions\n")

# for t in tests:
#     predict(t)

for t in tests:
    print(f"{t:25s} -> {predict(t)}") # space with 25 characters


Predictions

I                         -> love
I love                    -> doing
I love doing              -> mathematics
calculus                  -> is
calculus is               -> great
calculus is great         -> mathematics
mathematics               -> is
mathematics is            -> quite
mathematics is quite      -> useful


In [ ]:
tests=[
    "I",
    "I love",
    "I love doing",
    "calculus",
    "calculus is",
    "calculus is great",
    "mathematics",
    "mathematics is",
    "mathematics is quite"
]

print("\nPredictions\n")

# for t in tests:
#     predict(t)

for t in tests:
    print(f"{t:25s} -> {predict(t)}")


Predictions

I                         -> love
I love                    -> doing
I love doing              -> mathematics
calculus                  -> is
calculus is               -> great
calculus is great         -> mathematics
mathematics               -> is
mathematics is            -> quite
mathematics is quite      -> useful


## Transposition and Matrix multiplication

In [ ]:
tar = torch.rand(3,5,7)

In [ ]:
tar.shape

torch.Size([3, 5, 7])

In [ ]:
tar.transpose(-2,-1).shape

torch.Size([3, 7, 5])

In [ ]:
tar.transpose(0,2).shape

torch.Size([7, 5, 3])

In [ ]:
tar.shape

torch.Size([3, 5, 7])

In [ ]:
nar = torch.randn(3, 7, 2)

In [ ]:
tar.shape, nar.shape

(torch.Size([3, 5, 7]), torch.Size([3, 7, 2]))

In [ ]:
torch.matmul(tar, nar).shape

torch.Size([3, 5, 2])

### Position Embedding

In [ ]:
import math
import torch

In [ ]:
seq_len = 5
d_model = 12

pe = torch.zeros(seq_len, d_model)

In [ ]:
pe.shape

torch.Size([5, 12])

In [ ]:
position = torch.arange(seq_len).unsqueeze(1)

In [ ]:
position

tensor([[0],
        [1],
        [2],
        [3],
        [4]])

In [ ]:
torch.arange(0, d_model, 2)

tensor([ 0,  2,  4,  6,  8, 10])

In [ ]:
math.log(10000.0) / d_model

0.7675283643313486

In [ ]:
range_pow = torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)

In [ ]:
range_pow

tensor([-0.0000, -1.5351, -3.0701, -4.6052, -6.1402, -7.6753])

In [ ]:
div = torch.exp(range_pow)

In [ ]:
div

tensor([1.0000e+00, 2.1544e-01, 4.6416e-02, 1.0000e-02, 2.1544e-03, 4.6416e-04])

In [ ]:
position

tensor([[0],
        [1],
        [2],
        [3],
        [4]])

In [ ]:
position * div

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 2.1544e-01, 4.6416e-02, 1.0000e-02, 2.1544e-03, 4.6416e-04],
        [2.0000e+00, 4.3089e-01, 9.2832e-02, 2.0000e-02, 4.3089e-03, 9.2832e-04],
        [3.0000e+00, 6.4633e-01, 1.3925e-01, 3.0000e-02, 6.4633e-03, 1.3925e-03],
        [4.0000e+00, 8.6177e-01, 1.8566e-01, 4.0000e-02, 8.6177e-03, 1.8566e-03]])

In [ ]:
pe

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [ ]:
pe[:, 0::2] = torch.sin(position * div)

In [ ]:
pe.round(decimals=2)

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.8400,  0.0000,  0.2100,  0.0000,  0.0500,  0.0000,  0.0100,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.9100,  0.0000,  0.4200,  0.0000,  0.0900,  0.0000,  0.0200,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000],
        [ 0.1400,  0.0000,  0.6000,  0.0000,  0.1400,  0.0000,  0.0300,  0.0000,
          0.0100,  0.0000,  0.0000,  0.0000],
        [-0.7600,  0.0000,  0.7600,  0.0000,  0.1800,  0.0000,  0.0400,  0.0000,
          0.0100,  0.0000,  0.0000,  0.0000]])

In [ ]:
pe[:, 0::2] = torch.sin(position * div)
pe[:, 1::2] = torch.cos(position * div)

pe = pe.unsqueeze(0)